###### Patricia Clarji 202500382 | Marie-Lie kadado 202402112

# Diabetes Readmission Prediction

## Objectives
- **Multiclass Classification:** Predict the patient's readmission category (`readmitted`)
- **Regression:** Predict time spent in hospital (`time_in_hospital`)

Dataset:
Diabetes 130-US Hospitals Dataset

### Project Description

Hospital readmissions are costly and may indicate complications in patient care.
The goal of this project is to analyze hospital records of diabetic patients and build machine learning models for:

- **Multiclass classification:** predict the patient's readmission category (`NO`, `>30`, `<30`)
- **Regression:** predict the patient's length of stay in the hospital (`time_in_hospital`)

The dataset contains hospital records of diabetic patients from 1999–2008 across 130 US hospitals.

### Step-1- Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.patches import Rectangle


### Step-2- Load DataSet

In [ ]:
#Config:
DATA_PATH = "../data/diabetes.csv"
CLASSIFICATION_TARGET = "readmitted"
REGRESSION_TARGET = "time_in_hospital"
RANDOM_STATE = 42
TARGET= 'readmitted'

In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH, na_values='?')

columns = df.columns

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Split Categorical and Numerical Columns
categorical_cols = [
    'race', 'gender', 'age', 'weight',
    'payer_code', 'medical_specialty',
    'diag_1', 'diag_2', 'diag_3',
    'max_glu_serum', 'A1Cresult',
    'metformin', 'repaglinide', 'nateglinide',
    'chlorpropamide', 'glimepiride', 'acetohexamide',
    'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone', 'change', 'diabetesMed',
    'readmitted'
]

numerical_cols = [
    'encounter_id', 'patient_nbr',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

### Step 3 - Dataset Overview

#### 3.1 Head (5 rows)

In [ ]:
print("Dataset preview:")
display(df.head(10))


#### 3.2 Info

In [ ]:
print("\nDataset info:")
df.info()

#### 3.3 Missing values

In [ ]:
print("\nMissing values per column:")
display(df.isnull().sum().sort_values(ascending=False))

#### 3.4 Numerical cols

In [ ]:
print("\nNumerical summary:")
display(df[numerical_cols].describe().round(2))

#### 3.5 Categorical cols

In [ ]:
print("\nCategorical summary:")
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False))
    print("-" * 40)

#### 3.6 Other

In [ ]:
print("\nShape of dataset:", df.shape)
print("\nNumber of numeric columns:", len(numerical_cols))
print("\nNumber of categorical columns:", len(categorical_cols))
print("\nClassification target:", CLASSIFICATION_TARGET)
print("\nRegression target:", REGRESSION_TARGET)

#### Step 4- Data Visualization
In this step, we explore the dataset using visualizations to identify patterns, trends, and relationships between variables. This helps in understanding the data before applying preprocessing and machine learning models.

#### 4.0 Visualization SetUp

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

readmit_order = ['NO', '>30', '<30']
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
            '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']

num_cols = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_medications',
    'number_outpatient',
    'number_inpatient'
]

utilization_cols = [
    'number_outpatient',
    'number_emergency',
    'number_inpatient'
]

corr_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

#### 4.1 Target Variable Analysis
In this section, we analyze the target variables of the project: readmitted for the classification task and time_in_hospital for the regression task. This helps us understand class balance, distribution shape, and potential modeling challenges.

##### A- Readmission Distribution

In [ ]:
plt.figure(figsize=(8, 5))

ax = sns.countplot(data=df, x='readmitted', order=readmit_order)

total = len(df)
for p in ax.patches:
    if isinstance(p, Rectangle):
        count = int(p.get_height())
        pct = 100 * count / total
        ax.annotate(
            f"{count}\n({pct:.1f}%)",
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha='center',
            va='bottom',
            fontsize=10
        )

plt.title("Distribution of Readmission Status")
plt.xlabel("Readmission Category")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()

#### Interpretation
Most patients were not readmitted, while the <30 class is the smallest. This shows class imbalance, which matters for the classification task.

#### B- time_in_hospital distribution
Next, we analyze the regression target time_in_hospital to understand its distribution, central tendency, and potential skewness or outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x='time_in_hospital', bins=14, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Time in Hospital")
axes[0].set_xlabel("Days in Hospital")
axes[0].set_ylabel("Frequency")

sns.boxplot(data=df, x='time_in_hospital', ax=axes[1])
axes[1].set_title("Boxplot of Time in Hospital")
axes[1].set_xlabel("Days in Hospital")

plt.tight_layout()
plt.show()

##### Interpretation

Most hospital stays are short, concentrated around a few days(peek btw 3-4 days), while a smaller number of cases have much longer stays. The boxplot also shows some outliers.

#### 4.2 Univariate Analysis

##### A- Numerical Features Distributions

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

##### B- Numerical Feature Outlier

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x=col, ax=axes[i])
    axes[i].set_title(f"Boxplot of {col}")
    axes[i].set_xlabel(col)

fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

##### C- Categorical Features Distribution

In [ ]:
cat_cols = ['gender', 'race', 'age']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = age_order if col == 'age' else df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

#### Interpretation
Patients who are readmitted tend to have longer hospital stays, more medications, and more prior inpatient visits. These variables strongly indicate patient complexity and are likely to be important predictors.

#### 4.3 Bivariate Analysis: Features vs Readmission

##### A- Demographic Features

In [ ]:
age_readmit = pd.crosstab(df['age'], df['readmitted'], normalize='index').reindex(age_order)
gender_order = df['gender'].value_counts().index
gender_readmit = pd.crosstab(df['gender'], df['readmitted'], normalize='index').reindex(gender_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

age_readmit[readmit_order].plot(kind='bar', stacked=True, ax=axes[0])
axes[0].set_title("Readmission Proportion by Age Group")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Readmitted')

gender_readmit[readmit_order].plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_title("Readmission Proportion by Gender")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Proportion")
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Readmitted')

plt.tight_layout()
plt.show()

#### Interpretation
Older age groups show higher readmission proportions, suggesting age may be an important predictor. Gender differences are minimal, indicating that gender alone may not strongly influence readmission.

##### B- Clinical Features

In [ ]:
a1c_order = df['A1Cresult'].value_counts().index
a1c_readmit = pd.crosstab(df['A1Cresult'], df['readmitted'], normalize='index').reindex(a1c_order)

glu_order = df['max_glu_serum'].value_counts().index
glu_readmit = pd.crosstab(df['max_glu_serum'], df['readmitted'], normalize='index').reindex(glu_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

a1c_readmit[readmit_order].plot(kind='bar', stacked=True, ax=axes[0])
axes[0].set_title("Readmission Proportion by A1C Result")
axes[0].set_xlabel("A1C Result")
axes[0].set_ylabel("Proportion")
axes[0].legend(title='Readmitted')

glu_readmit[readmit_order].plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_title("Readmission Proportion by Max Glucose Serum")
axes[1].set_xlabel("Max Glucose Serum")
axes[1].set_ylabel("Proportion")
axes[1].legend(title='Readmitted')

plt.tight_layout()
plt.show()

#### Interpretation
Patients with abnormal A1C and glucose results tend to have higher readmission proportions. This suggests that poor glycemic control is associated with increased readmission risk.

#### C- Hospital Utilization Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

sns.boxplot(data=df, x='readmitted', y='time_in_hospital', order=readmit_order, ax=axes[0])
axes[0].set_title("Time in Hospital by Readmission Class")
axes[0].set_xlabel("Readmission Category")
axes[0].set_ylabel("Days in Hospital")

sns.boxplot(data=df, x='readmitted', y='num_medications', order=readmit_order, ax=axes[1])
axes[1].set_title("Number of Medications by Readmission Class")
axes[1].set_xlabel("Readmission Category")
axes[1].set_ylabel("Number of Medications")

sns.boxplot(data=df, x='readmitted', y='number_inpatient', order=readmit_order, ax=axes[2])
axes[2].set_title("Previous Inpatient Visits by Readmission Class")
axes[2].set_xlabel("Readmission Category")
axes[2].set_ylabel("Number of Inpatient Visits")

sns.boxplot(data=df, x='readmitted', y='num_lab_procedures', order=readmit_order, ax=axes[3])
axes[3].set_title("Lab Procedures by Readmission Class")
axes[3].set_xlabel("Readmission Category")
axes[3].set_ylabel("Number of Lab Procedures")

plt.tight_layout()
plt.show()

#### Interpretation
Patients who are readmitted tend to have longer hospital stays, use more medications, and have more previous inpatient visits. These patterns suggest that hospital utilization and clinical complexity are strongly related to readmission risk.

#### 4.4. Missingness Overview

##### A- Missing percentage by feature

In [ ]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(10, 6))
sns.barplot(x=missing_pct.values, y=missing_pct.index)
plt.title("Missing Values Percentage by Feature")
plt.xlabel("Missing Percentage")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

##### B- Missingness heatmap for columns with missing values

In [ ]:
missing_cols = df.columns[df.isnull().any()].tolist()

plt.figure(figsize=(12, 6))
sns.heatmap(df[missing_cols].isnull(), cbar=False)
plt.title("Missingness Heatmap")
plt.xlabel("Features with Missing Values")
plt.ylabel("Observations")
plt.tight_layout()
plt.show()

#### 4.5 Correlation and Numerical Relationships

##### A- Correlation heatmap

In [ ]:
corr = df[corr_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()

#### Interpretation

The scatterplots show moderate positive relationships between several numerical features. In particular, patients with more diagnoses and lab procedures tend to receive more medications. While these relationships are not extremely strong, they indicate that combinations of these variables may be useful for predicting readmission. No severe linear relationships are observed, suggesting that multicollinearity may not be a major issue.

##### B- Selected numerical relationship plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(data=df, x='num_lab_procedures', y='num_medications', alpha=0.3, ax=axes[0])
axes[0].set_title("Lab Procedures vs Medications")

sns.scatterplot(data=df, x='number_inpatient', y='time_in_hospital', alpha=0.3, ax=axes[1])
axes[1].set_title("Previous Inpatient Visits vs Hospital Stay")

sns.scatterplot(data=df, x='number_diagnoses', y='num_medications', alpha=0.3, ax=axes[2])
axes[2].set_title("Diagnoses vs Medications")

plt.tight_layout()
plt.show()

#### Interpretation

The scatterplots show moderate positive relationships between several numerical features. In particular, patients with more diagnoses and lab procedures tend to receive more medications. While these relationships are not extremely strong, they indicate that combinations of these variables may be useful for predicting readmission. No severe linear relationships are observed, suggesting that multicollinearity may not be a major issue.

#### 4.6 Additional Categorical Analysis

##### Readmission rate by top race categories

In [ ]:
top_race = df['race'].value_counts().index
race_readmit = pd.crosstab(df['race'], df['readmitted'], normalize='index').reindex(top_race)

plt.figure(figsize=(8, 5))
race_readmit[readmit_order].plot(kind='bar', stacked=True)
plt.title("Readmission Proportion by Race")
plt.xlabel("Race")
plt.ylabel("Proportion")
plt.xticks(rotation=45)
plt.legend(title='Readmitted')
plt.tight_layout()
plt.show()

##### Readmission rate by admission type

In [ ]:
admission_order = df['admission_type_id'].value_counts().index
admission_readmit = pd.crosstab(df['admission_type_id'], df['readmitted'], normalize='index').reindex(admission_order)

plt.figure(figsize=(8, 5))
admission_readmit[readmit_order].plot(kind='bar', stacked=True)
plt.title("Readmission Proportion by Admission Type")
plt.xlabel("Admission Type ID")
plt.ylabel("Proportion")
plt.legend(title='Readmitted')
plt.tight_layout()
plt.show()

### Step 5 - Data Preprocessing

Data preprocessing prepares the dataset for machine learning by handling
missing values, removing irrelevant features, encoding categorical variables,
and scaling numerical features.

#### 5.1 Missing Values Check

In [ ]:
print("Missing values per column:")
display(df.isnull().sum().sort_values(ascending=False))

##### 5.1.1 Handling missing values


In [ ]:
# 1. Remove rows with missing target
df = df.dropna(subset=['readmitted'])

# 2. Diagnosis columns -> keep missing as category
diag_cols = ['diag_1', 'diag_2', 'diag_3']
for col in diag_cols:
    df[col] = df[col].fillna('Missing').astype(str)

# 3. Lab result columns -> missing means not measured
lab_cols = ['max_glu_serum', 'A1Cresult']
for col in lab_cols:
    df[col] = df[col].fillna('NotMeasured')

# 4. Medication columns -> missing means No
medication_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

for col in medication_cols:
    df[col] = df[col].fillna('No')

# 5. Binary columns
df['change'] = df['change'].fillna('No')
df['diabetesMed'] = df['diabetesMed'].fillna('No')

# 6. Hospital code columns should be categorical
coded_cols = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
for col in coded_cols:
    df[col] = df[col].astype(str)

# 7. Numeric columns
median_cols = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_diagnoses'
]

zero_cols = [
    'number_outpatient',
    'number_emergency',
    'number_inpatient'
]

for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

for col in zero_cols:
    df[col] = df[col].fillna(0)

print("Remaining missing values:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

#### 5.2 Drop Columns with Too Many Missing Values

In [ ]:
cols_to_drop = ['weight', 'payer_code', 'medical_specialty']
df.drop(columns=cols_to_drop, inplace=True)

print("Dataset shape after dropping columns:", df.shape)

#### 5.3 Remove Duplicate Rows

In [ ]:
print("Duplicate rows before:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Duplicate rows after:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)

#### 5.4 Clean Invalid Gender Values

In [ ]:
print("Gender values before cleaning:")
print(df['gender'].value_counts(dropna=False))

df = df[df['gender'].isin(['Male', 'Female'])]

print("\nGender values after cleaning:")
print(df['gender'].value_counts(dropna=False))

#### 5.5 Check Multiclass Target Distribution

In [ ]:
print("Readmission classes:")
print(df['readmitted'].value_counts(dropna=False))

#### 5.6 Convert Age Groups to Numeric Values

In [ ]:
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}

df['age_numeric'] = df['age'].map(age_map)

display(df[['age', 'age_numeric']].head())

#### 5.7 Identify ID Columns

In [ ]:
id_cols = ['encounter_id', 'patient_nbr']
print("ID columns:", id_cols)

df.drop(columns=id_cols, inplace=True)

print("Dropped ID columns.")

#### 5.8 Final Missing Values Check

In [ ]:
print("Remaining missing values:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

#### 5.9 Preprocessing Summary

In [ ]:
print("Final dataset shape:", df.shape)
print("Remaining missing values total:", df.isnull().sum().sum())